<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/pnad/PNAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PNAD Contínua — desocupação, participação, ocupação e rendimento

Este notebook é o motor de coleta e publicação das séries `pnad-desocupacao`,
`pnad-participacao`, `pnad-nivel-ocupacao` e `pnad-rendimento-real`.

- **Fonte:** IBGE — Pesquisa Nacional por Amostra de Domicílios Contínua mensal
  (PNAD Contínua), divulgação por trimestre móvel, Brasil, pessoas de 14 anos ou mais.
  Tabelas SIDRA **6381** (taxa de desocupação, variável 4099), **5944** (taxa de
  participação, variável 4096), **6379** (nível de ocupação, variável 4097) e
  **6390** (rendimento médio mensal real, habitualmente recebido em todos os trabalhos,
  variável 5933).
- **Endpoint:** `https://apisidra.ibge.gov.br/values/t/<tabela>/n1/all/v/<variavel>/p/all?formato=json`
- **Definição:** cada ponto é o **trimestre móvel de três meses encerrado no mês** da data
  (C15: primeiro dia do mês de encerramento). Não é a taxa do mês isolado.
  - desocupação: desocupadas ÷ força de trabalho × 100;
  - participação: força de trabalho ÷ pessoas de 14 anos ou mais × 100;
  - nível de ocupação: ocupadas ÷ pessoas de 14 anos ou mais × 100;
  - rendimento: média mensal real (já deflacionada pelo INPC), em reais, do rendimento
    habitualmente recebido em todos os trabalhos pelas pessoas ocupadas com rendimento.
- **Unidade:** as três taxas em `%`, 1 casa decimal (a fonte publica 1 casa); o rendimento
  em `R$`, inteiro (a fonte publica reais sem centavos). `eixo: percentual` nas taxas e
  `reais` no rendimento; `medida: nivel`, `granularidade: mensal`.
- **Período publicado:** de `2012-03` (trimestre jan–fev–mar/2012, primeiro da PNAD
  Contínua mensal) até o último trimestre móvel fechado.
- **Arquivos gerados:** `indicadores/pnad/pnad-desocupacao.json`,
  `indicadores/pnad/pnad-participacao.json`, `indicadores/pnad/pnad-nivel-ocupacao.json`,
  `indicadores/pnad/pnad-rendimento-real.json`.

## O que este número não é

- **Não é o mês calendário.** A divulgação mensal da PNAD Contínua é um trimestre móvel:
  o valor de `2026-09` cobre jul–ago–set/2026. O IBGE é explícito: os indicadores mensais
  "não refletem a situação de cada mês, mas, sim, a situação do trimestre móvel".
- **Não é o Caged.** O Caged publica o estoque de vínculos formais; a PNAD mede a força de
  trabalho inteira (ocupadas e desocupadas, inclusive informalidade).
- **O rendimento não é renda domiciliar per capita.** É o rendimento do trabalho das
  pessoas ocupadas com rendimento, não a renda de todos os moradores dividida por eles. E é
  a série **habitual** — o que a pessoa costuma receber por mês —, não a efetivamente
  recebida no período. A PNAD também publica a efetiva (tabela 6387); ela não é usada aqui.
- **Não começa em 1995.** A PNAD Contínua mensal começa em `2012-03`; a PME (2002–2012)
  tinha outra metodologia e cobria só 6 regiões metropolitanas — não é emendada.
- **A taxa pode cair por saída da força de trabalho.** Por isso a série de participação
  acompanha a de desocupação: a identidade `nível de ocupação = participação ×
  (1 − desocupação/100)` é conferida a cada execução, com a tolerância do arredondamento
  de 1 casa (0,15 p.p.).


In [1]:
from datetime import date
from pathlib import Path
import json
import math
import os
import time

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "pnad"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ
DESTINO = Path(os.environ.get("PNAD_SAIDA", PASTA))


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name


BASE = "https://apisidra.ibge.gov.br/values"
SERIES = (
    {
        "id": "pnad-desocupacao",
        "tabela": "6381",
        "variavel": "4099",
        "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 1},
        "eixo": "percentual",
    },
    {
        "id": "pnad-participacao",
        "tabela": "5944",
        "variavel": "4096",
        "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 1},
        "eixo": "percentual",
    },
    {
        "id": "pnad-nivel-ocupacao",
        "tabela": "6379",
        "variavel": "4097",
        "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 1},
        "eixo": "percentual",
    },
    {
        "id": "pnad-rendimento-real",
        "tabela": "6390",
        "variavel": "5933",
        "unidade": {"tipo": "moeda", "simbolo": "R$", "casas": 0},
        "eixo": "reais",
    },
)
HOJE = date.today()
MES_CORRENTE = f"{HOJE:%Y%m}"


In [2]:
def baixar(tabela, variavel):
    url = f"{BASE}/t/{tabela}/n1/all/v/{variavel}/p/all?formato=json"
    for tentativa in range(1, 6):
        try:
            resposta = requests.get(url, timeout=60)
            resposta.raise_for_status()
            dados = resposta.json()
            if not isinstance(dados, list) or len(dados) < 2:
                raise ValueError("resposta sem linhas de dados")
            if dados[0].get("V") != "Valor":
                raise ValueError(f"cabeçalho inesperado: {dados[0]!r}")
            return dados, url
        except (requests.RequestException, ValueError):
            if tentativa == 5:
                raise
            time.sleep(2 ** tentativa)


def extrair(dados):
    """Devolve {AAAAMM: valor} e {AAAAMM: rótulo do trimestre móvel}."""
    valores = {}
    rotulos = {}
    for linha in dados[1:]:
        periodo = linha["D3C"]
        valor = linha["V"]
        if valor in ("...", "-", "X", ""):
            continue
        valores[periodo] = float(valor)
        rotulos[periodo] = linha.get("D3N", periodo)
    if not valores:
        raise ValueError("série vazia")
    return valores, rotulos


COLETADOS = {}
for item in SERIES:
    dados, url = baixar(item["tabela"], item["variavel"])
    valores, rotulos = extrair(dados)
    fechados = {periodo: valor for periodo, valor in valores.items() if periodo < MES_CORRENTE}
    COLETADOS[item["id"]] = {
        "tabela": item["tabela"],
        "url": url,
        "valores": fechados,
        "rotulos": rotulos,
    }
    ultimo = max(fechados)
    print(f"{item['id']}: {len(fechados)} trimestres móveis, "
          f"{min(fechados)} a {ultimo} ({rotulos[ultimo]})")


pnad-desocupacao: 175 trimestres móveis, 201203 a 202609 (jul-ago-set 2026)


pnad-participacao: 175 trimestres móveis, 201203 a 202609 (jul-ago-set 2026)


pnad-nivel-ocupacao: 175 trimestres móveis, 201203 a 202609 (jul-ago-set 2026)


pnad-rendimento-real: 175 trimestres móveis, 201203 a 202609 (jul-ago-set 2026)


In [3]:
coberturas = {item["id"]: set(COLETADOS[item["id"]]["valores"]) for item in SERIES}
assert len({frozenset(periodos) for periodos in coberturas.values()}) == 1, (
    "as quatro séries têm coberturas diferentes: "
    + ", ".join(f"{k}: {min(v)}–{max(v)}" for k, v in coberturas.items())
)
PERIODOS = sorted(coberturas[SERIES[0]["id"]])
print(f"{len(PERIODOS)} trimestres móveis em comum: {PERIODOS[0]} a {PERIODOS[-1]}")


175 trimestres móveis em comum: 201203 a 202609


In [4]:
def formatar(periodo):
    return f"{periodo[:4]}-{periodo[4:]}-01"


def montar(item):
    coleta = COLETADOS[item["id"]]
    pontos = [
        {"data": formatar(periodo), "valor": coleta["valores"][periodo]}
        for periodo in PERIODOS
    ]
    return {
        "versao": 1,
        "id": item["id"],
        "unidade": item["unidade"],
        "eixo": item["eixo"],
        "medida": "nivel",
        "granularidade": "mensal",
        "fonte": f"IBGE (PNAD Contínua, SIDRA {item['tabela']})",
        "urlFonte": coleta["url"],
        "dados": pontos,
    }


DESTINO.mkdir(parents=True, exist_ok=True)
SERIES_JSON = {}
for item in SERIES:
    serie = montar(item)
    caminho = DESTINO / f"{item['id']}.json"
    with caminho.open("w", encoding="utf-8") as arquivo:
        json.dump(serie, arquivo, ensure_ascii=False, indent=2)
    SERIES_JSON[item["id"]] = serie
    print(f"gravado {relativo(caminho)}: {len(serie['dados'])} pontos")


gravado pnad-desocupacao.json: 175 pontos
gravado pnad-participacao.json: 175 pontos
gravado pnad-nivel-ocupacao.json: 175 pontos
gravado pnad-rendimento-real.json: 175 pontos


## Validação inline

A célula falha se o contrato, a ordem, a unicidade, a continuidade mensal, a finitude, a
igualdade com a fonte, a identidade entre as três taxas ou os números citados em
`docs/metodologia.md` forem violados. A conferência do rendimento é contra a fonte e contra
a publicação externa registrada na metodologia; o notebook não busca fonte terceira.


In [5]:
CHAVES = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
ESPERADO = {
    "pnad-desocupacao": ("6381", {"tipo": "percentual", "simbolo": "%", "casas": 1}, "percentual"),
    "pnad-participacao": ("5944", {"tipo": "percentual", "simbolo": "%", "casas": 1}, "percentual"),
    "pnad-nivel-ocupacao": ("6379", {"tipo": "percentual", "simbolo": "%", "casas": 1}, "percentual"),
    "pnad-rendimento-real": ("6390", {"tipo": "moeda", "simbolo": "R$", "casas": 0}, "reais"),
}

for item in SERIES:
    serie = SERIES_JSON[item["id"]]
    tabela, unidade, eixo = ESPERADO[item["id"]]
    assert set(serie) == CHAVES, sorted(set(serie) ^ CHAVES)
    assert serie["versao"] == 1, serie["versao"]
    assert serie["id"] == item["id"], serie["id"]
    assert serie["unidade"] == unidade, serie["unidade"]
    assert serie["eixo"] == eixo, serie["eixo"]
    assert serie["medida"] == "nivel", serie["medida"]
    assert serie["granularidade"] == "mensal", serie["granularidade"]
    assert serie["fonte"] == f"IBGE (PNAD Contínua, SIDRA {tabela})", serie["fonte"]
    assert serie["urlFonte"] == COLETADOS[item["id"]]["url"], serie["urlFonte"]

    dados = serie["dados"]
    assert dados, f"{item['id']}: série vazia"
    datas = [ponto["data"] for ponto in dados]
    dias = [date.fromisoformat(texto) for texto in datas]
    for ponto in dados:
        assert set(ponto) == {"data", "valor"}, (item["id"], sorted(ponto))
        assert isinstance(ponto["data"], str), ponto
        assert not isinstance(ponto["valor"], bool), ponto
        assert isinstance(ponto["valor"], (int, float)), ponto
        assert math.isfinite(ponto["valor"]), ponto

    assert all(dia.day == 1 for dia in dias), f"{item['id']}: dia diferente de 01"
    assert all(a < b for a, b in zip(dias, dias[1:])), f"{item['id']}: datas fora de ordem"
    assert all(
        (b.year * 12 + b.month) - (a.year * 12 + a.month) == 1
        for a, b in zip(dias, dias[1:])
    ), f"{item['id']}: mês faltando ou repetido"
    assert datas[0] == "2012-03-01", (item["id"], datas[0])
    assert datas[-1] == formatar(PERIODOS[-1]), (item["id"], datas[-1])
    assert len(dados) == len(PERIODOS), (item["id"], len(dados))

    esperado = [
        {"data": formatar(periodo), "valor": COLETADOS[item["id"]]["valores"][periodo]}
        for periodo in PERIODOS
    ]
    assert dados == esperado, f"{item['id']}: valor publicado difere da fonte"

    with (DESTINO / f"{item['id']}.json").open(encoding="utf-8") as arquivo:
        assert json.load(arquivo) == serie, f"{item['id']}: arquivo gravado difere do objeto"

# Identidade: nível de ocupação = participação × (1 − desocupação/100), com o arredondamento
# de 1 casa da fonte. O resíduo máximo medido em 2026-11 é 0,1125 p.p. (2019-02).
por_data = {
    item["id"]: {ponto["data"]: ponto["valor"] for ponto in SERIES_JSON[item["id"]]["dados"]}
    for item in SERIES
}
residuos = []
for data in por_data["pnad-desocupacao"]:
    desocupacao = por_data["pnad-desocupacao"][data]
    participacao = por_data["pnad-participacao"][data]
    nivel = por_data["pnad-nivel-ocupacao"][data]
    calculado = participacao * (1 - desocupacao / 100)
    residuos.append((abs(calculado - nivel), data, participacao, desocupacao, nivel, calculado))
residuos.sort(reverse=True)
assert residuos[0][0] <= 0.15, residuos[:3]
print(f"identidade nível = participação × (1 − desocupação): resíduo máximo "
      f"{residuos[0][0]:.4f} p.p. em {residuos[0][1]}")


# Números citados em docs/metodologia.md (publicação de 2026-11-03, rendimento em 2026-11-04).
def valor(id_serie, data):
    return por_data[id_serie][data]


assert valor("pnad-desocupacao", "2012-03-01") == 8.0
assert valor("pnad-desocupacao", "2020-09-01") == 14.9
assert valor("pnad-desocupacao", "2025-12-01") == 5.1
assert valor("pnad-desocupacao", "2026-09-01") == 5.2
assert valor("pnad-participacao", "2012-03-01") == 62.3
assert valor("pnad-participacao", "2019-06-01") == 63.6
assert valor("pnad-participacao", "2020-07-01") == 56.5
assert valor("pnad-participacao", "2026-09-01") == 62.2
assert valor("pnad-nivel-ocupacao", "2012-03-01") == 57.3
assert valor("pnad-nivel-ocupacao", "2020-08-01") == 48.4
assert valor("pnad-nivel-ocupacao", "2025-11-01") == 59.0
assert valor("pnad-nivel-ocupacao", "2026-09-01") == 59.0
assert valor("pnad-rendimento-real", "2012-03-01") == 3179.0
assert valor("pnad-rendimento-real", "2021-12-01") == 3074.0
assert valor("pnad-rendimento-real", "2026-03-01") == 3811.0
assert valor("pnad-rendimento-real", "2026-09-01") == 3790.0

# O rendimento real ainda não recuperou o pico de 2026-03 no último trimestre fechado,
# e o piso da série é a pandemia: o sinal tem de refletir os dois fatos.
assert valor("pnad-rendimento-real", "2026-09-01") < valor("pnad-rendimento-real", "2026-03-01")
assert valor("pnad-rendimento-real", "2021-12-01") < valor("pnad-rendimento-real", "2012-03-01")

print(
    f"ok: {len(PERIODOS)} trimestres móveis, {formatar(PERIODOS[0])} a {formatar(PERIODOS[-1])}; "
    "quatro séries idênticas à fonte, identidade das taxas conferida"
)


identidade nível = participação × (1 − desocupação): resíduo máximo 0.1125 p.p. em 2019-02-01
ok: 175 trimestres móveis, 2012-03-01 a 2026-09-01; quatro séries idênticas à fonte, identidade das taxas conferida
